**Document Ingestion Pipeline**

In [2]:
# Import Libraries
import os
from pathlib import Path

import fitz
import spacy
from docx import Document

***Configuration Variables***

In [14]:
# Preset Variables configs for use throughout
DATA_DIRECTORY = Path("docs")

CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

SUPPORTED_EXTENSIONS = {".pdf", ".docx"}

In [4]:
# Initialize spaCy w/ Sentencizer for faster and more efficient retrieval (by sentence vs full doc)
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

In [5]:
# Sentencizer Sanity Check
test_text = ("This is a single sentence about something." " We have another sentence here as well." "Let's write a third sentence, so that we may test stuff.")

test_doc = nlp(test_text)
test_sentences = [sent.text.strip() for sent in test_doc.sents]
for i, sentence in enumerate(test_sentences, 1):
    print(f"Chunk {i}: {sentence}")


Chunk 1: This is a single sentence about something.
Chunk 2: We have another sentence here as well.
Chunk 3: Let's write a third sentence, so that we may test stuff.


In [6]:
# Create Text Cleaning Function to remove unnecessary whitespaces but preserve punc/nums/stop words as source material
def clean_text(text):
    text = text.strip()
    text = " ".join(text.split())

    return text

In [20]:
# Create PDF Parsing function using PyMuPDF
def parse_pdf(pdf_path):
    # Open out PDF file from path
    doc = fitz.open(pdf_path)
    # Extract PDF Metadata
    metadata = {"filename": Path(pdf_path).name,"title": doc.metadata.get("title", ""), "author": doc.metadata.get("author", ""), "total_pages": len(doc), "file_type": "pdf"}

    print("Document Information")
    print(f"Total Pages: {len(doc)}")
    print(f"Metadata: {doc.metadata}\n")

    pages = []
    # Go through each page of PDF
    for page_num, page in enumerate(doc, start=1):

        text = clean_text(page.get_text("text"))

        # Skip blank pages
        if not text:
            continue

        pages.append({"text": text,"metadata": {"filename": metadata["filename"], "page_number": page_num}})

    doc.close()

    return pages, metadata

In [22]:
# Creat DOCX Parsing function using 
def parse_docx(docx_path):
    # Open out DOCX file from path
    doc = Document(docx_path)
    # Extract DOCX Metadata (Note: python-docx iterates through paragraphs instead of pages so metadata changed accordingly)
    metadata = {"filename": Path(docx_path).name,"title": doc.core_properties.title or "", "author": doc.core_properties.author or "", "total_paragraphs": len(doc.paragraphs), "file_type": "docx"}

    print("Document Information")
    print(f"Total Paragraphs: {len(doc.paragraphs)}")
    print(f"Metadata: {metadata}\n")

    paragraphs = []
    # Go through each paragraph of DOCX
    for paragraph_num, paragraph in enumerate(doc.paragraphs, start=1):

        text = clean_text(paragraph.text)

        # Skip blank pages
        if not text:
            continue

        paragraphs.append({"text": text,"metadata": {"filename": metadata["filename"], "paragraph_number": paragraph_num}, "paragraph_style": paragraph.style.name})


    return paragraphs, metadata

In [9]:
# Parse Doc based on Filetype, Return Error if wrong file type used
def parse_document(file_path):
    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return parse_pdf(file_path)

    elif extension == ".docx":
        return parse_docx(file_path)

    else:
        raise ValueError(f"Unsupported file type: {extension}")

In [23]:
# Test Parse on PDF
parse_document("docs/MDPs___An_Introduction.pdf")

Document Information
Total Pages: 4
Metadata: {'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.26', 'creationDate': 'D:20250225234619Z', 'modDate': 'D:20250225234619Z', 'trapped': '', 'encryption': None}



([{'text': 'Introduction to Markov Decision Processes in Reinforcement Learning 1 Introduction Reinforcement Learning (RL) is a framework where an agent learns to make decisions by inter- acting with an environment. At the heart of many RL problems is the Markov Decision Process (MDP), which provides a simple yet powerful way to model decision-making in uncertain situations. This tutorial is designed to explain the basics of MDPs in an accessible manner, without heavy mathematical notation, and to show how they underpin many RL methods. 2 What is a Markov Decision Process? An MDP is a mathematical model that describes an environment in which decisions are made in steps. It is defined by a few simple components: • States (S): All possible situations the agent might encounter. For example, in a game, a state might describe the current position of the player. • Actions (A): The set of choices available to the agent in each state. In our game example, actions could be moving left, right, o

In [24]:
# Test Parse on DOCX
parse_document("docs/Group03_Week9_Proposal_CRS.docx")

Document Information
Total Paragraphs: 54
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'title': '', 'author': 'Claudia Porto', 'total_paragraphs': 54, 'file_type': 'docx'}



([{'text': 'Sean Costello, Kat Fountain, Claudia Porto, Christopher Swartz',
   'metadata': {'filename': 'Group03_Week9_Proposal_CRS.docx',
    'paragraph_number': 1},
   'paragraph_style': 'Normal'},
  {'text': 'IE7374 - Group 03',
   'metadata': {'filename': 'Group03_Week9_Proposal_CRS.docx',
    'paragraph_number': 2},
   'paragraph_style': 'Normal'},
  {'text': 'Milestone 2: Project Proposal',
   'metadata': {'filename': 'Group03_Week9_Proposal_CRS.docx',
    'paragraph_number': 3},
   'paragraph_style': 'Normal'},
  {'text': 'What Up Doc? A Privacy-First Local RAG Application',
   'metadata': {'filename': 'Group03_Week9_Proposal_CRS.docx',
    'paragraph_number': 4},
   'paragraph_style': 'Normal'},
  {'text': '1. Final Topic Area & Model Selection',
   'metadata': {'filename': 'Group03_Week9_Proposal_CRS.docx',
    'paragraph_number': 5},
   'paragraph_style': 'Normal'},
  {'text': 'This project operates in the NLP domain, specifically retrieval-augmented generation (RAG). NLP wa

*** Next we Need to Tokenize the Text with spaCy ***

In [25]:
def tokenize_text(text):
    doc = nlp.make_doc(text)
    return [token.text for token in doc]

def count_tokens(text):
    return len(tokenize_text(text))



In [27]:
# Test Tokenizer/Counter

test_sample = "These are word! Let's count them."
test_tokens = tokenize_text(test_sample)

print(test_tokens)
print("Token Count:", count_tokens(test_sample))

['These', 'are', 'word', '!', 'Let', "'s", 'count', 'them', '.']
Token Count: 9


*** Build the Chunking Section, this breaks our docs down into smaller pieces**

In [30]:
# Fixed Chunking Strategy
def fixed_chunking(records, chunk_size = CHUNK_SIZE, overlap = CHUNK_OVERLAP):
    # Make sure overlap isnt larger thn Chunk Size
    if overlap >= chunk_size:
        raise ValueError("Overlap too Large.")
    
    chunks = []
    chunk_index = 0

    step_size = chunk_size - overlap

    for record in records:
        text = record["text"]
        original_metadata = record["metadata"]
        tokens = tokenize_text(text)

        for start in range(0, len(tokens), step_size):
            end = start + chunk_size
            chunk_tokens = tokens[start:end]

            if not chunk_tokens:
                continue

            chunk_text = " ".join(chunk_tokens)

            chunk_metadata = original_metadata.copy()
            chunk_metadata.update({"chunk_index": chunk_index, "chunking_strategy": "fixed", "token_count": len(chunk_tokens)})
            chunks.append({"text": chunk_text, "metadata": chunk_metadata})
            
            chunk_index += 1
    return chunks

In [31]:
# Test of Fixed Chunking
records, metadata = parse_document("docs/Group03_Week9_Proposal_CRS.docx")

test_chunks = fixed_chunking(records,chunk_size=20, overlap=5)

Document Information
Total Paragraphs: 54
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'title': '', 'author': 'Claudia Porto', 'total_paragraphs': 54, 'file_type': 'docx'}



In [33]:
# Test Results
print(f"Total chunks: {len(test_chunks)}")

for chunk in test_chunks[:3]:
    print(chunk["metadata"])
    print(chunk["text"])
    print("-" * 60)

Total chunks: 98
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 1, 'chunk_index': 0, 'chunking_strategy': 'fixed', 'token_count': 11}
Sean Costello , Kat Fountain , Claudia Porto , Christopher Swartz
------------------------------------------------------------
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 2, 'chunk_index': 1, 'chunking_strategy': 'fixed', 'token_count': 4}
IE7374 - Group 03
------------------------------------------------------------
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 3, 'chunk_index': 2, 'chunking_strategy': 'fixed', 'token_count': 5}
Milestone 2 : Project Proposal
------------------------------------------------------------


In [34]:
# Create Sentence Chunking Function using spaCy
def sentence_chunking(records, chunk_size=CHUNK_SIZE):

    chunks = []
    chunk_index = 0

    # Go through each parsed record
    for record in records:

        text = record["text"]
        metadata = record["metadata"]

        doc = nlp(text)

        current_chunk = []
        current_tokens = 0

        # Go through each sentence
        for sentence in doc.sents:

            sentence = sentence.text.strip()

            if not sentence:
                continue

            sentence_tokens = count_tokens(sentence)

            # Add sentence if it still fits
            if current_tokens + sentence_tokens <= chunk_size:

                current_chunk.append(sentence)
                current_tokens += sentence_tokens

            # Otherwise save current chunk and start a new one
            else:

                chunk_text = " ".join(current_chunk)

                chunk_metadata = metadata.copy()
                chunk_metadata.update({"chunk_index": chunk_index,"chunking_strategy": "sentence","token_count": current_tokens})

                chunks.append({"text": chunk_text,"metadata": chunk_metadata})

                chunk_index += 1

                current_chunk = [sentence]
                current_tokens = sentence_tokens

        # Save the final chunk
        if current_chunk:

            chunk_text = " ".join(current_chunk)

            chunk_metadata = metadata.copy()
            chunk_metadata.update({"chunk_index": chunk_index,"chunking_strategy": "sentence","token_count": current_tokens})

            chunks.append({"text": chunk_text,"metadata": chunk_metadata})

            chunk_index += 1

    return chunks

In [35]:
# Test the Sentence Chunking
records, metadata = parse_document("docs/Group03_Week9_Proposal_CRS.docx")
sentence_chunks = sentence_chunking(records, chunk_size=50)

Document Information
Total Paragraphs: 54
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'title': '', 'author': 'Claudia Porto', 'total_paragraphs': 54, 'file_type': 'docx'}



In [36]:
print("Document Metadata:")
print(metadata)
print("\nTotal Parsed Records:", len(records))
print("Total Sentence Chunks:", len(sentence_chunks))

for chunk in sentence_chunks[:5]:
    print("Metadata:", chunk["metadata"])
    print("Text:", chunk["text"])
    print("-" * 60)

Document Metadata:
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'title': '', 'author': 'Claudia Porto', 'total_paragraphs': 54, 'file_type': 'docx'}

Total Parsed Records: 46
Total Sentence Chunks: 53
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 1, 'chunk_index': 0, 'chunking_strategy': 'sentence', 'token_count': 11}
Text: Sean Costello, Kat Fountain, Claudia Porto, Christopher Swartz
------------------------------------------------------------
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 2, 'chunk_index': 1, 'chunking_strategy': 'sentence', 'token_count': 4}
Text: IE7374 - Group 03
------------------------------------------------------------
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 3, 'chunk_index': 2, 'chunking_strategy': 'sentence', 'token_count': 5}
Text: Milestone 2: Project Proposal
------------------------------------------------------------
Metadata: {'filename': 'Group03_W

In [37]:
# Create Paragraph Chunking Function
def paragraph_chunking(records, chunk_size=CHUNK_SIZE):

    chunks = []
    chunk_index = 0

    current_chunk = []
    current_tokens = 0

    # Go through each parsed paragraph (or page for PDFs)
    for record in records:

        paragraph = record["text"]
        metadata = record["metadata"]

        paragraph_tokens = count_tokens(paragraph)

        # Add paragraph if it still fits
        if current_tokens + paragraph_tokens <= chunk_size:

            current_chunk.append(paragraph)
            current_tokens += paragraph_tokens

        # Otherwise save current chunk and start a new one
        else:

            chunk_text = "\n\n".join(current_chunk)

            chunk_metadata = metadata.copy()
            chunk_metadata.update({"chunk_index": chunk_index, "chunking_strategy": "paragraph", "token_count": current_tokens})

            chunks.append({"text": chunk_text,"metadata": chunk_metadata})

            chunk_index += 1

            current_chunk = [paragraph]
            current_tokens = paragraph_tokens

    # Save the final chunk
    if current_chunk:

        chunk_text = "\n\n".join(current_chunk)

        chunk_metadata = metadata.copy()
        chunk_metadata.update({"chunk_index": chunk_index, "chunking_strategy": "paragraph", "token_count": current_tokens})

        chunks.append({"text": chunk_text, "metadata": chunk_metadata})

    return chunks

In [40]:
# Testing Paragraph Chunking
records, metadata = parse_document("docs/Group03_Week9_Proposal_CRS.docx")
paragraph_chunks = paragraph_chunking(records, chunk_size=100)

Document Information
Total Paragraphs: 54
Metadata: {'filename': 'Group03_Week9_Proposal_CRS.docx', 'title': '', 'author': 'Claudia Porto', 'total_paragraphs': 54, 'file_type': 'docx'}



In [39]:
print("Total Paragraph Chunks:", len(paragraph_chunks))

for chunk in paragraph_chunks[:5]:
    print(chunk["metadata"])
    print(chunk["text"])
    print("-" * 70)

Total Paragraph Chunks: 15
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 6, 'chunk_index': 0, 'chunking_strategy': 'paragraph', 'token_count': 39}
Sean Costello, Kat Fountain, Claudia Porto, Christopher Swartz

IE7374 - Group 03

Milestone 2: Project Proposal

What Up Doc? A Privacy-First Local RAG Application

1. Final Topic Area & Model Selection
----------------------------------------------------------------------
{'filename': 'Group03_Week9_Proposal_CRS.docx', 'paragraph_number': 7, 'chunk_index': 1, 'chunking_strategy': 'paragraph', 'token_count': 62}
This project operates in the NLP domain, specifically retrieval-augmented generation (RAG). NLP was chosen because the core problem of enabling secure, source-grounded question answering over private documents is fundamentally a language task, and it aligns with our team’s collective background in software engineering, data analytics, and computer science.
-------------------------------------------------------

*** Chunk Strategy Selection and Total Doc Ingestion Method ***

In [41]:
def chunk_document(records, strategy):
    if strategy == "fixed":
        return fixed_chunking(records)

    elif strategy == "sentence":
        return sentence_chunking(records)

    elif strategy == "paragraph":
        return paragraph_chunking(records)

    else:
        raise ValueError(f"Not a valdi chunking strategy: {strategy}")

In [43]:
def ingest_document(file_path, strategy):
    records, metadata = parse_document(file_path)

    chunks = chunk_document(records, strategy)

    return chunks, metadata